In [1]:
import pandas as pd
import logging
import sys
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv

In [2]:
# === НАСТРОЙКА ПУТЕЙ ===
current_dir = Path.cwd()
project_src = current_dir.parent

if str(project_src) not in sys.path:
    sys.path.append(str(project_src))

from app.rag_pipeline import ScienceRAG

# Загружаем переменные окружения
load_dotenv()

True

In [3]:
# === КОНФИГУРАЦИЯ ===
INPUT_FILE = Path('../../data/eval/gd.json')
OUTPUT_FILE = Path('../../data/eval/gd_filled.parquet')
# Сколько чанков будет возвращать ретривер?
TOP_K = 10

# Подавляем INFO логи от httpx
logging.getLogger("httpx").setLevel(logging.WARNING)

In [4]:
# === ЗАГРУЗКА ДАННЫХ ===
df = pd.read_json(INPUT_FILE)

In [5]:
# === ИНИЦИАЛИЗАЦИЯ RAG ===
rag = ScienceRAG()

INFO:app.rag_pipeline:Initializing RAG on device: CUDA
INFO:app.rag_pipeline:Connected to Qdrant collection: nlp2025_chunks
INFO:app.rag_pipeline:Loading retriever: Qwen/Qwen3-Embedding-0.6B...
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: Qwen/Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

INFO:sentence_transformers.SentenceTransformer:1 prompt is loaded, with the key: query
INFO:app.rag_pipeline:Retriever loaded on CPU
INFO:app.rag_pipeline:Loading LLM: Qwen/Qwen2.5-3B-Instruct...
INFO:app.rag_pipeline:Using 4-bit quantization (NF4)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

INFO:app.rag_pipeline:LLM loaded on CUDA
INFO:app.rag_pipeline:VRAM allocated: 2.07 GB
INFO:app.rag_pipeline:VRAM reserved: 6.12 GB
INFO:app.rag_pipeline:RAG system is ready.



In [7]:
# === ФУНКЦИЯ ЗАПОЛНЕНИЯ ===
def get_rag_data(question, top_k):
    """Получает ответ и контекст от RAG системы"""
    res = rag.answer(question, top_k)
    context_list = [source['text'] for source in res.get('sources', [])]
    return pd.Series([res['answer'], context_list])

In [8]:
# === ОБРАБОТКА ДАТАСЕТА ===
tqdm.pandas()
df[['answer', 'retrieved_contexts']] = df['question'].progress_apply(get_rag_data, top_k=TOP_K)

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [23:21<00:00, 28.03s/it]


In [9]:
# === СОХРАНЕНИЕ РЕЗУЛЬТАТА ===
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_FILE, index=False)

print(f"\nДатасет сохранён: {OUTPUT_FILE}")


Датасет сохранён: ..\..\data\eval\gd_filled.parquet


In [10]:
df

,question,ground_truth,ground_context,answer,retrieved_contexts
0,Why is the reliance on context considered a si...,The interpretation of sarcasm depends on unsta...,[Large Language Models for Subjective Language...,The reliance on context is considered a signif...,"[In this section, we will focus on the develop..."
1,What sources were used to create the HQ-GCM-RA...,The dataset was curated from ancient Chinese m...,[Hengqin-RA-v1: Advanced Large Language Model ...,The HQ-GCM-RA-C1 dataset for the Hengqin-RA-v1...,"[So in this paper, we proposed HQ-GCM-RA-C1, t..."
2,What barriers prevent experts outside the AI f...,Barriers include a lack of technical expertise...,[A Survey of Large Language Models in Discipli...,Experts outside the AI field face several barr...,[#### 5.1.1 Quality of Discipline-related data...
3,What specific prompt is inserted between neura...,The prompt inserted is: “decode the above neur...,[decoding inner speech with an end-to-end brai...,The specific prompt inserted between neural an...,"[We propose a unified brain decoder, as illust..."
4,How is the BERT architecture modified to perfo...,A classification layer is added on top of the ...,[PLEX: Perturbation-free Local Explanations fo...,During the fine-tuning process for sentiment c...,"[In this work, BERT is fine-tuned on polarity ..."
5,What specific Indonesian term yielded an inacc...,"The token ""cie"" returned an inaccurate gloss. ...",[Context-Aware Pragmatic Metacognitive Prompti...,Based on the information provided in Document ...,[We analyzed the actual definitions returned b...
6,What specific attention mechanism does the DeB...,DeBERTa implements a disentangled attention me...,[Classification of Hope in Textual Data using ...,The DeBERTa architecture employs a disentangle...,[This section details our comprehensive approa...
7,What specific architectural modification is ad...,A classification layer is added on top of the ...,[PLEX: Perturbation-free Local Explanations fo...,When fine-tuning BERT for sentiment classifica...,"[In this work, BERT is fine-tuned on polarity ..."
8,In the context of text refinement for binary c...,A counterfactual text is a generated refinemen...,[Refining Financial Consumer Complaints throug...,In the context of text refinement for binary c...,[Table 8: LLM-judge prompt for evaluating the ...
9,What primary barriers prevent researchers from...,"Barriers include limited data availability, in...",[Zero-shot generation of synthetic neurosurgic...,Primary barriers preventing researchers from e...,[Neurosurgical research relies on data collect...
